In [273]:
"""
This module is used for reading the data and creating the test and training
data sets.
"""
import urllib.request 

def readData(url):
    """
    This function reads all the data contained in the given url.  It creates a dictionary for each line and adds this dictionary to a list.

    Parameters: 
        url: name of the url containing the data records

        Returns: a list of dictionaries
    """
    
    # This function is done - Do not modify.
    dataSet = []

    #Open the web page containing the data
    response = urllib.request.urlopen(url)
    
    #Read first line and remove new line indicator
    html = response.readline().rstrip()
    
    # Read in file
    while len(html) != 0:
        #data will contain a list with each element a different attribute
        lineList = html.decode('utf-8').split(",")

        # Create a dictionary for the line
        #   assigns each attribute of the record (each item in the linelist)
        #   to an element of the dictionary, using the constant keys
        record = {}
        record["age"] = float(lineList[0])
        record["workclass"] = lineList[1]
        record["educationnum"] = float(lineList[4])
        record["marital"] = lineList[5]
        record["occupation"] = lineList[6]
        record["relationship"] = lineList[7]
        record["race"] = lineList[8]
        record["sex"] = lineList[9]
        record["capitalgain"] = float(lineList[10])
        record["capitalloss"] = float(lineList[11])
        record["hours"] = float(lineList[12])
        record["class"] = lineList[14]

        # Add the dictionary to a list
        dataSet.append(record)

        #Read next line
        html = response.readline().rstrip()

    return dataSet


In [274]:
def makeTrainingSet(data):
    """"
    Makes a new list of dictionaries containing the first half of the records
    from the full data set (the parameter "data")
    
    Parameters:
        data - a list of dictionaries read from the url
    Returns:
        trainingData - a list of dictionaries (1/2 of data)
    """
    #take the length of half the training set
    halfLength = int(len(data)/2)
    #take rows up to half of the dataset for the training set  
    trainingData = data[:halfLength]
   
    return trainingData


In [345]:
def makeTestSet(data):
    """
    Create a test set of data from the second half of the data (ie. the
    second half of the data set (indicated by the parameter data).
    Add to each dictionary in this set a key called "predicted"
    and set it to "unknown".
    Paramters:
        data - a list of dictionaries (complete data read from url)
    Returns:
        testData - a list of dictionaries (second half of data file)
    """
    #take the length of half the training set
    halfLength = int(len(data)/2)
    
    #take rows starting at the halfway mark until the end
    testData = data[halfLength:]
    
    #add a "predicted" key to each dictionary and set to "unknown"
    for item in testData:
        item.update({"predicted":"unknown"})
    
    return testData


In [346]:
#testing goes here.  For each module you should show tests for each function.
if __name__ == "__main__":
    data = readData("http://research.cs.queensu.ca/home/cords2/annualIncome.txt")
    #trying to print all the data will probably make your program crash.
    #so, print the first and last values to check them.
    print(data[0])
    print(data[len(data)-1])
    print(len(data), "records have been read")
    
    #see what hald of the set would look like
    print ("Half of the records would be", int(len(data)/2))

    #check the length of the training data and print a sample
    trainingData = makeTrainingSet(data)
    print("The length of the training set is", len(trainingData))
    print(trainingData[0])
    
    #check the length of the test data and print a sample
    testData = makeTestSet(data)
    print("the length of the test set is", len(testData))
    print(testData[0], "\n")
    
    #see if the test and training sets add to the total set
    if len(trainingData)+len(testData) == len(data):
        print("The datasets are complete :)", "\n")
    else:
        print("something is wrong :(", "\n")
        
    #check that the test set includes the added 'predicted' key
    if 'predicted' in testData[0]:
        print("'predicted' key was found in dictionary")
    else:
        print("'predicted' key not found")
    

{'age': 25.0, 'workclass': 'Private', 'educationnum': 7.0, 'marital': 'Never-married', 'occupation': 'Machine-op-inspct', 'relationship': 'Own-child', 'race': 'Black', 'sex': 'Male', 'capitalgain': 0.0, 'capitalloss': 0.0, 'hours': 40.0, 'class': '<=50K'}
{'age': 35.0, 'workclass': 'Self-emp-inc', 'educationnum': 13.0, 'marital': 'Married-civ-spouse', 'occupation': 'Exec-managerial', 'relationship': 'Husband', 'race': 'White', 'sex': 'Male', 'capitalgain': 0.0, 'capitalloss': 0.0, 'hours': 60.0, 'class': '>50K'}
15060 records have been read
Half of the records would be 7530
The length of the training set is 7530
{'age': 25.0, 'workclass': 'Private', 'educationnum': 7.0, 'marital': 'Never-married', 'occupation': 'Machine-op-inspct', 'relationship': 'Own-child', 'race': 'Black', 'sex': 'Male', 'capitalgain': 0.0, 'capitalloss': 0.0, 'hours': 40.0, 'class': '<=50K'}
the length of the test set is 7530
{'age': 66.0, 'workclass': 'Private', 'educationnum': 9.0, 'marital': 'Married-civ-spouse

In [393]:
def buildClassifier(trainingData):
    """
    Create a classifier which takes in a training set and builds
    a model for >50K income and <=50K income by calculating the 
    averages of the various attributes
    Parameters:
        trainingData - a list of dictonaries created from half
                       the dataset
    Returns:
        classifier - 
    """
    #define what are the numerical and categorical keys
    numerical = ["age", "educationnum", "capitalgain", "capitalloss", "hours"]
    categorical = ["workclass", "marital", "occupation", "relationship", "race", "sex"]

    #set blank dictionary for class <50K
    moreThan50K = {
        'age': 0,
        'workclass': {'Private' : 0, 'Self-emp-not-inc' : 0, 'Self-emp-inc' : 0, 'Federal-gov' : 0, 
                      'Local-gov' : 0, 'State-gov' : 0, 'Without-pay' : 0, 'Never-worked' : 0},
        'educationnum': 0,
        'marital': {'Married-civ-spouse' : 0, 'Divorced' : 0, 'Never-married' : 0, 'Separated' : 0, 'Widowed' : 0, 
                    'Married-spouse-absent' : 0, 'Married-AF-spouse' : 0},
        'occupation' : {'Tech-support' : 0, 'Craft-repair' : 0, 'Other-service' : 0, 'Sales' : 0, 'Exec-managerial' : 0
                        ,'Prof-specialty' : 0, 'Handlers-cleaners' : 0, 'Machine-op-inspct' : 0, 'Adm-clerical' : 0, 
                        'Farming-fishing' : 0, 'Transport-moving' : 0, 'Priv-house-serv': 0, 'Protective-serv' : 0
                        , 'Armed-Forces' : 0},
        'relationship' : {'Wife' : 0, 'Own-child' : 0, 'Husband' : 0, 'Not-in-family' : 0, 'Other-relative' : 0
                          , 'Unmarried' : 0},
        'race' : {'White' : 0, 'Asian-Pac-Islander' : 0, 'Amer-Indian-Eskimo' : 0, 'Other' : 0, 'Black' : 0},
        'sex' : {'Female' : 0, 'Male' : 0},
        'capitalgain' : 0.0,
        'capitalloss': 0.0,
        'hours': 0
    }

    #set blank dictionary for class <=50K
    lessThan50K = {
        'age': 0,
        'workclass': {'Private' : 0, 'Self-emp-not-inc' : 0, 'Self-emp-inc' : 0, 'Federal-gov' : 0, 
                      'Local-gov' : 0, 'State-gov' : 0, 'Without-pay' : 0, 'Never-worked' : 0},
        'educationnum': 0,
        'marital': {'Married-civ-spouse' : 0, 'Divorced' : 0, 'Never-married' : 0, 'Separated' : 0, 'Widowed' : 0, 
                    'Married-spouse-absent' : 0, 'Married-AF-spouse' : 0},
        'occupation' : {'Tech-support' : 0, 'Craft-repair' : 0, 'Other-service' : 0, 'Sales' : 0, 'Exec-managerial' : 0
                        ,'Prof-specialty' : 0, 'Handlers-cleaners' : 0, 'Machine-op-inspct' : 0, 'Adm-clerical' : 0, 
                        'Farming-fishing' : 0, 'Transport-moving' : 0, 'Priv-house-serv': 0, 'Protective-serv' : 0
                        , 'Armed-Forces' : 0},
        'relationship' : {'Wife' : 0, 'Own-child' : 0, 'Husband' : 0, 'Not-in-family' : 0, 'Other-relative' : 0
                          , 'Unmarried' : 0},
        'race' : {'White' : 0, 'Asian-Pac-Islander' : 0, 'Amer-Indian-Eskimo' : 0, 'Other' : 0, 'Black' : 0},
        'sex' : {'Female' : 0, 'Male' : 0},
        'capitalgain' : 0.0,
        'capitalloss': 0.0,
        'hours': 0
    }

    k = 0
    j = 0

    for i in range(len(trainingData)): #iter over all line
        data = trainingData[i] 
        d = data['class']
        if d == '>50K':
            k += 1
            for key, val in data.items():
                if key in numerical:
                    moreThan50K[key] += float(val)
                if key in categorical:
                    moreThan50K[key][val] += 1

        else:
            j += 1
            for key, val in data.items():
                if key in numerical:
                    lessThan50K[key] += float(val)
                if key in categorical:
                    lessThan50K[key][val] += 1

    for key in moreThan50K:  
        if key in numerical:
            moreThan50K[key] =  round(moreThan50K[key]/k, 5)
            lessThan50K[key] =  round(lessThan50K[key]/j, 5)
        else:
            k1 = key
            for key in moreThan50K[k1]:
                moreThan50K[k1][key] =  round(moreThan50K[k1][key]/k, 5)
                lessThan50K[k1][key] =  round(lessThan50K[k1][key]/j, 5)
            
    
    return moreThan50K, lessThan50K

if __name__ == "__main__":
    #testing for classifier 
    data = readData("http://research.cs.queensu.ca/home/cords2/annualIncome.txt")
    trainingData = makeTrainingSet(data)

    moreThan50K, lessThan50K = buildClassifier(trainingData)
    print (moreThan50K, "\n")
    print (lessThan50K, "\n")
    print('Percent sex adds to a total of =', (moreThan50K['sex']['Female'] + moreThan50K['sex']['Male'])*100, '%')


{'age': 44.17968, 'workclass': {'Private': 0.63845, 'Self-emp-not-inc': 0.09121, 'Self-emp-inc': 0.08902, 'Federal-gov': 0.04806, 'Local-gov': 0.07865, 'State-gov': 0.05461, 'Without-pay': 0.0, 'Never-worked': 0.0}, 'educationnum': 11.53905, 'marital': {'Married-civ-spouse': 0.85035, 'Divorced': 0.05298, 'Never-married': 0.06663, 'Separated': 0.01092, 'Widowed': 0.01147, 'Married-spouse-absent': 0.00601, 'Married-AF-spouse': 0.00164}, 'occupation': {'Tech-support': 0.03168, 'Craft-repair': 0.12452, 'Other-service': 0.01638, 'Sales': 0.12616, 'Exec-managerial': 0.25177, 'Prof-specialty': 0.24194, 'Handlers-cleaners': 0.01584, 'Machine-op-inspct': 0.03277, 'Adm-clerical': 0.07373, 'Farming-fishing': 0.01693, 'Transport-moving': 0.04042, 'Priv-house-serv': 0.00055, 'Protective-serv': 0.02622, 'Armed-Forces': 0.00109}, 'relationship': {'Wife': 0.0852, 'Own-child': 0.01147, 'Husband': 0.76188, 'Not-in-family': 0.10759, 'Other-relative': 0.00437, 'Unmarried': 0.02949}, 'race': {'White': 0.90

In [384]:
def testAccuracy(testData, moreThan50K, lessThan50K):
    """
    Using the testing set, test to see if the accuracy of the classifier
    by using the models for >50K and <=50K to classify the test data.
    Then, compare the predictions to the real values. 
    Parameters:
        testData - half of the original data set aside for testing 
        moreThan50K - model with averages for data over 50K
        lessThan50K - model with averages for data less than or equal to 50K
    Returns:
        numCorrect - number of correctly classified sets
    """
    numerical = ["age", "educationnum", "capitalgain", "capitalloss", "hours"]
    categorical = ["workclass", "marital", "occupation", "relationship", "race", "sex"]

    for i in range(len(testData)):
        less = 0
        more = 0
        data = testData[i]
        for key, val in data.items():
            if key in numerical:
                if abs(val - moreThan50K[key]) > abs(val - lessThan50K[key]):
                    less += 1
                else:
                    more += 1
            if key in categorical:
                k1 = key
                data1 = data[k1]
                if moreThan50K[k1][data1] < lessThan50K[k1][data1]:
                    less += 1
                else:
                    more += 1
        if less < more:
            testData[i]['predicted'] = '>50K'
        else:
            testData[i]['predicted'] = '<=50K'

    numCorrect = 0

    for i in range(len(testData)):
        if testData[i]['predicted'] == testData[i]['class']:
            numCorrect += 1
        
    return numCorrect

if __name__ == "__main__":
    #testing for classifier 
    data = readData("http://research.cs.queensu.ca/home/cords2/annualIncome.txt")
    testData = makeTestSet(data)
    trainingData = makeTrainingSet(data)
    moreThan50K, lessThan50K = buildClassifier(trainingData) 
    numCorrect = testAccuracy(testData, moreThan50K, lessThan50K) 
    print (testData[0], '\n')
    print ('>50K =', moreThan50K, "\n")
    print ('<=50K =', lessThan50K, "\n")


    # looking at it manually
    # age = >50K
    # workclass = <=50K
    # eductionnum = <=50K
    # marital = >50K
    # occupation = <=50K
    # relationship = >50K
    # race = >50K
    # sex = >50K
    # capitalgain = <=50K
    # capitalloss = <=50K
    # hours = <=50K

    # class = <=50K
    # <=50K  = 6
    # >50K = 5

    # predicted <=50K = GOOD ! (got this from looking within the function)



{'age': 66.0, 'workclass': 'Private', 'educationnum': 9.0, 'marital': 'Married-civ-spouse', 'occupation': 'Transport-moving', 'relationship': 'Husband', 'race': 'White', 'sex': 'Male', 'capitalgain': 0.0, 'capitalloss': 0.0, 'hours': 16.0, 'class': '<=50K', 'predicted': '<=50K'} 

>50K = {'age': 44.18, 'workclass': {'Private': 0.638, 'Self-emp-not-inc': 0.091, 'Self-emp-inc': 0.089, 'Federal-gov': 0.048, 'Local-gov': 0.079, 'State-gov': 0.055, 'Without-pay': 0.0, 'Never-worked': 0.0}, 'educationnum': 11.539, 'marital': {'Married-civ-spouse': 0.85, 'Divorced': 0.053, 'Never-married': 0.067, 'Separated': 0.011, 'Widowed': 0.011, 'Married-spouse-absent': 0.006, 'Married-AF-spouse': 0.002}, 'occupation': {'Tech-support': 0.032, 'Craft-repair': 0.125, 'Other-service': 0.016, 'Sales': 0.126, 'Exec-managerial': 0.252, 'Prof-specialty': 0.242, 'Handlers-cleaners': 0.016, 'Machine-op-inspct': 0.033, 'Adm-clerical': 0.074, 'Farming-fishing': 0.017, 'Transport-moving': 0.04, 'Priv-house-serv': 0.

In [418]:
def main():
    print('Reading in data')
    data = readData("http://research.cs.queensu.ca/home/cords2/annualIncome.txt")
    print('Making training and test files')
    testData = makeTestSet(data)
    trainingData = makeTrainingSet(data)
    print('Building classifier')
    moreThan50K, lessThan50K = buildClassifier(trainingData) 
    print('Classifying test data', '\n')
    numCorrect = testAccuracy(testData, moreThan50K, lessThan50K)
    
    Total = len(testData)
    numIncorrect = Total - numCorrect
    Accuracy = ((numCorrect/Total) * 100)
    
    print('Classified Correctly:', numCorrect)
    print('Classified Incorrectly:', numIncorrect)
    print('Accuracy:', '{:.2f}'.format(round((Accuracy), 2)),'%')
    
main()    
    

Reading in data
Making training and test files
Building classifier
Classifying test data 

Classified Correctly: 5926
Classified Incorrectly: 1604
Accuracy: 78.70 %
